# Single-Environment MTPPO Experiment

Smoke run replicating Lu et al. 2025 ('Enhanced MTPPO for IRP-VMI'):

- 20 retailers, 7 planning days;
- MTPPO with paper-faithful GIN + multi-head attention;
- GA(IRP) genetic-algorithm baseline (paper §5, Table 3);
- `greedy` control for fast reference.

OR-Tools is no longer used. All routing baselines are genetic algorithms, matching the paper.


In [ ]:
from __future__ import annotations

import asyncio
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

if sys.platform.startswith('win'):
    try:
        asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())
    except AttributeError:
        pass

ROOT = Path.cwd().resolve().parents[0]
SRC = ROOT / 'src'
OUTPUTS = ROOT / 'outputs'
OUTPUTS.mkdir(exist_ok=True)

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from psrp_mtppo.baselines import evaluate_baseline, run_baseline_episode
from psrp_mtppo.config import EnvironmentConfig, ModelConfig, TrainingConfig
from psrp_mtppo.experiments import (
    plot_episode_dashboard,
    plot_retailer_heatmaps,
    plot_training_dashboard,
    run_joint_experiment,
)
from psrp_mtppo.ga import GAConfig

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 140)


d:\PythonProjects\PSRP_D3PO\.venv311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Experiment Settings

Edit these values first. The notebook below uses them everywhere, so this is the main control panel for the experiment.


In [ ]:
SHOW_PROGRESS = True
SEED = 42
TRACE_SEED = 123
DEVICE = 'auto'          # 'auto' picks cuda when available, otherwise cpu
NUM_ENVS = 4             # parallel environment rollouts per training iteration

# Environment setup
NUM_RETAILERS = 20
HORIZON_DAYS = 7
HISTORY_LENGTH = 7
DEMAND_LOW = 1
DEMAND_HIGH = 50
RETAILER_CAPACITY = 100.0
INITIAL_RETAILER_INVENTORY = 100.0
SUPPLIER_INITIAL_INVENTORY = 1_000_000.0
VEHICLE_CAPACITY = 100.0
HOLDING_COST = 2.0
PRODUCT_PRICE = 20.0
SALES_LOSS_PENALTY = 0.3
TRANSPORT_COST_PER_DISTANCE = 1000.0

# Model setup (paper hyperparameters)
GIN_DIMS = (64, 128, 128)
MLP_DIMS = (128, 128)
STATE_EMBED_DIM = 128
ATTENTION_HEADS = 8
ATTENTION_LAYERS = 2
DROPOUT = 0.1

# Training setup (paper hyperparameters; TRAINING_ITERATIONS lowered for smoke)
LEARNING_RATE = 1e-3
GAMMA = 0.9
TRAIN_BATCH_SIZE = 256
MINIBATCH_SIZE = 64
PPO_EPOCHS = 4
TRAINING_ITERATIONS = 5
EVAL_EPISODES = 3
CLIP_PARAM = 0.2
VALUE_CLIP_PARAM = 0.1
KL_COEFFICIENT = 0.2
ENTROPY_COEFFICIENT = 0.01
MAX_GRAD_NORM = 1.0

# Baseline setup -- GA(IRP) is the paper's joint baseline; 'greedy' is a fast control.
BASELINE_NAME = 'ga_irp'   # one of: 'greedy', 'ga_inv', 'ga_vrp', 'ga_irp'
CONTROL_NAME = 'greedy'
GA_POPULATION = 40         # paper uses 100; lowered for smoke
GA_GENERATIONS = 15        # paper uses 30; lowered for smoke
GA_CROSSOVER = 0.6
GA_MUTATION = 0.05
GA_SELECTION_PRESSURE = 1.5
GA_PARENT_RATIO = 0.5

GA_CFG = GAConfig(
    population_size=GA_POPULATION,
    generations=GA_GENERATIONS,
    crossover_prob=GA_CROSSOVER,
    mutation_prob=GA_MUTATION,
    selection_pressure=GA_SELECTION_PRESSURE,
    parent_ratio=GA_PARENT_RATIO,
    seed=SEED,
)


In [ ]:
ENV_CONFIG = EnvironmentConfig(
    num_retailers=NUM_RETAILERS,
    horizon_days=HORIZON_DAYS,
    history_length=HISTORY_LENGTH,
    holding_cost=HOLDING_COST,
    retailer_capacity=RETAILER_CAPACITY,
    initial_retailer_inventory=INITIAL_RETAILER_INVENTORY,
    supplier_initial_inventory=SUPPLIER_INITIAL_INVENTORY,
    demand_low=DEMAND_LOW,
    demand_high=DEMAND_HIGH,
    product_price=PRODUCT_PRICE,
    sales_loss_penalty=SALES_LOSS_PENALTY,
    vehicle_capacity=VEHICLE_CAPACITY,
    transport_cost_per_distance=TRANSPORT_COST_PER_DISTANCE,
    seed=SEED,
)

MODEL_CONFIG = ModelConfig(
    gin_dims=GIN_DIMS,
    mlp_dims=MLP_DIMS,
    state_embed_dim=STATE_EMBED_DIM,
    attention_heads=ATTENTION_HEADS,
    attention_layers=ATTENTION_LAYERS,
    dropout=DROPOUT,
)

TRAIN_CONFIG = TrainingConfig(
    learning_rate=LEARNING_RATE,
    gamma=GAMMA,
    train_batch_size=TRAIN_BATCH_SIZE,
    minibatch_size=MINIBATCH_SIZE,
    ppo_epochs=PPO_EPOCHS,
    training_iterations=TRAINING_ITERATIONS,
    evaluation_episodes=EVAL_EPISODES,
    clip_param=CLIP_PARAM,
    value_clip_param=VALUE_CLIP_PARAM,
    kl_coefficient=KL_COEFFICIENT,
    entropy_coefficient=ENTROPY_COEFFICIENT,
    max_grad_norm=MAX_GRAD_NORM,
    num_envs=NUM_ENVS,
    device=DEVICE,
    seed=SEED,
)

config_rows = [
    ('env', 'num_retailers', NUM_RETAILERS),
    ('env', 'horizon_days', HORIZON_DAYS),
    ('env', 'demand_low', DEMAND_LOW),
    ('env', 'demand_high', DEMAND_HIGH),
    ('env', 'vehicle_capacity', VEHICLE_CAPACITY),
    ('env', 'transport_cost_per_distance', TRANSPORT_COST_PER_DISTANCE),
    ('model', 'gin_dims', GIN_DIMS),
    ('model', 'mlp_dims', MLP_DIMS),
    ('model', 'state_embed_dim', STATE_EMBED_DIM),
    ('model', 'attention_heads', ATTENTION_HEADS),
    ('model', 'dropout', DROPOUT),
    ('train', 'learning_rate', LEARNING_RATE),
    ('train', 'gamma', GAMMA),
    ('train', 'train_batch_size', TRAIN_BATCH_SIZE),
    ('train', 'minibatch_size', MINIBATCH_SIZE),
    ('train', 'ppo_epochs', PPO_EPOCHS),
    ('train', 'training_iterations', TRAINING_ITERATIONS),
    ('train', 'evaluation_episodes', EVAL_EPISODES),
    ('train', 'num_envs', NUM_ENVS),
    ('train', 'device', TRAIN_CONFIG.resolved_device()),
    ('train', 'seed', SEED),
    ('baseline', 'name', BASELINE_NAME),
    ('baseline', 'control', CONTROL_NAME),
    ('baseline', 'ga_population', GA_POPULATION),
    ('baseline', 'ga_generations', GA_GENERATIONS),
]
config_table = pd.DataFrame(config_rows, columns=['group', 'name', 'value'])
config_table


,group,name,value
0,env,num_retailers,20
1,env,horizon_days,7
2,env,demand_low,1
3,env,demand_high,50
4,env,vehicle_capacity,100.0
5,env,transport_cost_per_distance,1000.0
6,model,gin_dims,"(64, 128, 128)"
7,model,mlp_dims,"(128, 128)"
8,model,state_embed_dim,128
9,model,attention_heads,8


## Train MTPPO on the single 20-retailer setting

This is the main training run for the notebook.


In [ ]:
trainer, history = run_joint_experiment(
    env_config=ENV_CONFIG,
    model_config=MODEL_CONFIG,
    training_config=TRAIN_CONFIG,
    show_progress=SHOW_PROGRESS,
)
history.to_csv(OUTPUTS / 'single_env_history.csv', index=False)
history


D:\PythonProjects\PSRP_D3PO\src\psrp_mtppo\models\attention.py:10: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(
MTPPO train (20 retailers):   0%|          | 0/5 [00:12<?, ?it/s]


ValueError: Expected parameter loc (Tensor of shape (64, 20)) of distribution Normal(loc: torch.Size([64, 20]), scale: torch.Size([64, 20])) to satisfy the constraint Real(), but found invalid values:
tensor([[nan, nan, nan,  ..., nan, nan, nan],
        [nan, nan, nan,  ..., nan, nan, nan],
        [nan, nan, nan,  ..., nan, nan, nan],
        ...,
        [nan, nan, nan,  ..., nan, nan, nan],
        [nan, nan, nan,  ..., nan, nan, nan],
        [nan, nan, nan,  ..., nan, nan, nan]], grad_fn=<SqueezeBackward1>)

In [ ]:
history[['iteration', 'mean_return', 'eval_sum_cost', 'eval_fill_rate', 'inventory_actor_loss', 'routing_actor_loss', 'critic_loss']].round(3)


In [ ]:
plot_training_dashboard(history)
plt.show()


## Compare MTPPO against GA(IRP) and greedy on the same environment

Both baselines are evaluated on the same 20-retailer configuration, not across multiple scales.


In [ ]:
mtppo_episode_rows = []
for eval_seed in range(100, 100 + EVAL_EPISODES):
    episode_summary, _episode_trace = trainer.rollout_episode(seed=eval_seed, greedy=True, show_progress=False)
    mtppo_episode_rows.append(episode_summary)
mtppo_eval_df = pd.DataFrame(mtppo_episode_rows)

ga_eval_df = evaluate_baseline(
    ENV_CONFIG,
    baseline=BASELINE_NAME,
    episodes=EVAL_EPISODES,
    ga_config=GA_CFG,
)
control_eval_df = evaluate_baseline(
    ENV_CONFIG,
    baseline=CONTROL_NAME,
    episodes=EVAL_EPISODES,
)

comparison_mean = pd.DataFrame(
    [
        {'algorithm': 'MTPPO', **mtppo_eval_df.mean(numeric_only=True).to_dict()},
        {'algorithm': BASELINE_NAME.upper(), **ga_eval_df.mean(numeric_only=True).to_dict()},
        {'algorithm': CONTROL_NAME.upper(), **control_eval_df.mean(numeric_only=True).to_dict()},
    ]
).set_index('algorithm')

comparison_std = pd.DataFrame(
    [
        {'algorithm': 'MTPPO', **mtppo_eval_df.std(numeric_only=True).fillna(0.0).to_dict()},
        {'algorithm': BASELINE_NAME.upper(), **ga_eval_df.std(numeric_only=True).fillna(0.0).to_dict()},
        {'algorithm': CONTROL_NAME.upper(), **control_eval_df.std(numeric_only=True).fillna(0.0).to_dict()},
    ]
).set_index('algorithm')

comparison_mean.round(3)


In [ ]:
comparison_std.round(3)


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for axis, metric in zip(axes, ['reward', 'sum_cost', 'route_distance', 'fill_rate']):
    comparison_mean[metric].plot(kind='bar', ax=axis, rot=0, title=metric.replace('_', ' ').title())
    axis.grid(alpha=0.3)
fig.tight_layout()
plt.show()


## Detailed episode trace on a fixed seed

This section is useful for debugging behavior day by day on the exact same environment instance.


In [ ]:
mtppo_trace_summary, mtppo_trace = trainer.rollout_episode(
    seed=TRACE_SEED,
    greedy=True,
    show_progress=SHOW_PROGRESS,
)

baseline_trace_summary, baseline_trace = run_baseline_episode(
    ENV_CONFIG,
    baseline=BASELINE_NAME,
    seed=TRACE_SEED,
    ga_config=GA_CFG,
)

trace_summary = pd.DataFrame(
    [
        {'algorithm': 'MTPPO', **mtppo_trace_summary},
        {'algorithm': BASELINE_NAME.upper(), **baseline_trace_summary},
    ]
).set_index('algorithm')
trace_summary.round(3)


In [ ]:
plot_episode_dashboard(mtppo_trace['daily'], title_prefix='MTPPO')
plt.show()


In [ ]:
plot_episode_dashboard(baseline_trace['daily'], title_prefix=f'Baseline-{BASELINE_NAME}')
plt.show()


In [ ]:
mtppo_daily = mtppo_trace['daily'].copy().rename(columns=lambda column: f'mtppo_{column}' if column != 'day' else column)
baseline_daily = baseline_trace['daily'].copy().rename(columns=lambda column: f'baseline_{column}' if column != 'day' else column)
daily_compare = mtppo_daily.merge(baseline_daily, on='day', how='inner')

fig, axes = plt.subplots(2, 2, figsize=(16, 9))
axes[0, 0].plot(daily_compare['day'], daily_compare['mtppo_reward'], marker='o', label='MTPPO')
axes[0, 0].plot(daily_compare['day'], daily_compare['baseline_reward'], marker='s', label=BASELINE_NAME.upper())
axes[0, 0].set_title('Reward per day')
axes[0, 0].legend(); axes[0, 0].grid(alpha=0.3)

axes[0, 1].plot(daily_compare['day'], daily_compare['mtppo_inventory_cost'], marker='o', label='MTPPO')
axes[0, 1].plot(daily_compare['day'], daily_compare['baseline_inventory_cost'], marker='s', label=BASELINE_NAME.upper())
axes[0, 1].set_title('Inventory cost per day')
axes[0, 1].legend(); axes[0, 1].grid(alpha=0.3)

axes[1, 0].plot(daily_compare['day'], daily_compare['mtppo_route_cost'], marker='o', label='MTPPO')
axes[1, 0].plot(daily_compare['day'], daily_compare['baseline_route_cost'], marker='s', label=BASELINE_NAME.upper())
axes[1, 0].set_title('Route cost per day')
axes[1, 0].legend(); axes[1, 0].grid(alpha=0.3)

axes[1, 1].plot(daily_compare['day'], daily_compare['mtppo_fill_rate'], marker='o', label='MTPPO')
axes[1, 1].plot(daily_compare['day'], daily_compare['baseline_fill_rate'], marker='s', label=BASELINE_NAME.upper())
axes[1, 1].set_title('Fill rate per day')
axes[1, 1].legend(); axes[1, 1].grid(alpha=0.3)

fig.tight_layout()
plt.show()


## Retailer-level heatmaps

These plots help inspect which retailers are being replenished, how demand evolves, and where inventory remains high or low over time.


In [ ]:
plot_retailer_heatmaps(
    ending_inventory=mtppo_trace['ending_inventory'],
    replenishment=mtppo_trace['replenishment'],
    demand=mtppo_trace['demand'],
    title_prefix='MTPPO',
)
plt.show()


In [ ]:
plot_retailer_heatmaps(
    ending_inventory=baseline_trace['ending_inventory'],
    replenishment=baseline_trace['replenishment'],
    demand=baseline_trace['demand'],
    title_prefix=f'Baseline-{BASELINE_NAME}',
)
plt.show()


In [ ]:
comparison_mean.to_csv(OUTPUTS / 'single_env_comparison_mean.csv')
comparison_std.to_csv(OUTPUTS / 'single_env_comparison_std.csv')
mtppo_trace['daily'].to_csv(OUTPUTS / 'single_env_mtppo_daily_trace.csv', index=False)
baseline_trace['daily'].to_csv(OUTPUTS / 'single_env_baseline_daily_trace.csv', index=False)
print('Saved notebook artifacts to', OUTPUTS)
